In [ ]:
# Lab type: write
# Course: DS101 — Introduction to Data Science
# Lesson: Your First Machine Learning Model
# Task: Train and evaluate a linear regression model to predict house prices

# Lab: Your First Machine Learning Model

In this lab you'll train a linear regression model to predict house prices
from a small set of features. The workflow is the same as the lesson:
prepare → split → fit → predict → evaluate.

Unlike the lesson (one feature), you'll use **two features** here. That
means your feature matrix `X` will have two columns — the rest of the
code is identical.

## Step 1: Install and import

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt

## Step 2: Load the dataset

In [ ]:
# House price dataset: size_sqft, num_bedrooms, price_gbp
data = {
    "size_sqft": [450, 620, 780, 510, 830, 960, 1100, 420, 690, 870,
                  730, 580, 1020, 490, 760, 900, 640, 810, 550, 1150,
                  470, 700, 850, 530, 980, 610, 740, 880, 1050, 590],
    "num_bedrooms": [1, 2, 3, 1, 3, 4, 4, 1, 2, 3,
                     2, 2, 4, 1, 3, 3, 2, 3, 2, 5,
                     1, 2, 3, 2, 4, 2, 3, 3, 4, 2],
    "price_gbp": [195000, 285000, 370000, 220000, 395000, 460000, 530000,
                  180000, 310000, 415000, 340000, 265000, 490000, 205000,
                  355000, 435000, 295000, 385000, 245000, 560000, 210000,
                  325000, 405000, 250000, 470000, 280000, 350000, 420000,
                  505000, 270000]
}
df = pd.DataFrame(data)
df.head()

> **Question:** What are the ranges of each column? Run `df.describe()` to check.

<details>
<summary>🔑 Reveal answer — Q1</summary>

With the built-in dataset: `size_sqft` ranges from 450–1100, `num_bedrooms` from 1–4, `price_gbp` from roughly £190,000–£500,000. No negatives or zeros — the ranges are internally consistent.

`df.describe()` shows `count`, `mean`, `std`, `min`, `25%`, `50%`, `75%`, `max` for each numeric column. The std relative to the mean tells you how spread out values are; a high std in `price_gbp` (e.g., std > mean/2) is normal for property data.

</details>

In [ ]:
# Your code here


## Step 3: Prepare X and y

In [ ]:
# Use both size_sqft and num_bedrooms as features
X = df[["size_sqft", "num_bedrooms"]]
y = df["price_gbp"]

print("X shape:", X.shape)
print("y shape:", y.shape)

## Step 4: Split into train and test sets

In [ ]:
# Split with 20% held back for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

## Step 5: Fit the model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

# Print the learned coefficients
for feature, coef in zip(X.columns, model.coef_):
    print(f"{feature}: £{coef:,.0f} per unit")
print(f"Intercept: £{model.intercept_:,.0f}")

> **Question:** How much does the model think each extra square foot adds to price?
> How much does each extra bedroom add? Do these numbers seem reasonable?

<details>
<summary>🔑 Reveal answer — Q2</summary>

**Per-sqft coefficient:** Expect something in the range £200–£500 per square foot for this dataset. Each extra sqft adds that amount to the predicted price, holding bedrooms constant.

**Per-bedroom coefficient:** Often smaller in magnitude — around £10,000–£30,000 — or even negative in small samples. A negative bedroom coefficient can appear when larger homes (high sqft, higher price) happen to have fewer bedrooms than expected, causing the model to "correct" for the correlation between sqft and bedrooms.

**Reasonableness:** Compare the coefficients to your own market intuition. If price-per-sqft comes out at £50 or £5,000 the model has likely been given poorly scaled data or there's a sign error in y.

</details>

## Step 6: Predict and evaluate

In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE:  £{mae:,.0f}")
print(f"R²:   {r2:.2f}")

> **Question:** Is the MAE large or small relative to the typical house price in this dataset?
> What does the R² tell you about how well the two features explain price?

<details>
<summary>🔑 Reveal answer — Q3</summary>

**MAE in context:** Divide MAE by the mean of `y_test` to get a relative error percentage. An MAE of £20,000 on a £300,000 mean price is ~7%, which is reasonable for two features on a small dataset. An MAE of £80,000+ on the same data would suggest the model is not capturing the main price drivers.

**R² interpretation:** R² = 1.0 is a perfect fit; R² = 0.0 means the model does no better than predicting the mean for every house. An R² of 0.85+ means `size_sqft` and `num_bedrooms` together explain 85% of the variance in price. Anything below ~0.6 for this clean dataset would indicate a problem worth investigating (wrong features in X, or a data loading error).

</details>

## Step 7: Compare actual vs predicted

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred, alpha=0.7)
# A perfect model would have all points on the diagonal
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], "r--", label="Perfect predictions")
plt.xlabel("Actual price (£)")
plt.ylabel("Predicted price (£)")
plt.title("Actual vs. predicted house price")
plt.legend()
plt.tight_layout()
plt.show()

> **Question:** How close do the points cluster to the diagonal? Are there any predictions
> that are particularly far off? What might explain those errors?

<details>
<summary>🔑 Reveal answer — Q4</summary>

**Reading the plot:** Points on (or near) the diagonal mean the predicted price matches the actual price. Points above the diagonal are under-predicted (actual > predicted); points below are over-predicted.

**Outliers in small datasets:** With only 10 rows, a single unusual house (e.g., large sqft but low price due to condition, or small sqft but high price due to location) will stand out visibly. Linear regression has no way to account for such factors — they are reflected as residuals in the plot.

**What to do next:** If several points are consistently above or below the line in a pattern (e.g., all large houses are under-predicted), that signals a missing feature or a non-linear relationship that a simple linear model cannot capture.

</details>

## Extension: try with only one feature

Re-run the model using **only** `size_sqft` as your feature (a single-column X).
Compare the MAE and R² with the two-feature model.

Does adding `num_bedrooms` meaningfully improve performance?

In [ ]:
# Your code here
